In [1]:
%load_ext line_profiler

In [2]:
from ml_switching_reg_sim.data_creation import (
    MisclassificationCreator,
    UberDatasetCreator,
    UberDatasetCreatorHet,
)
from datetime import timedelta
import pandas as pd
import sys
import formulaic
import pyhdfe

sys.path.append("/Users/lordflaron/Documents/uganda-uber-paper/paper/scripts")
from regression import RegressionData
from ml_switching_reg.cm import cm_4, cm_10
from numba import jit

import autograd.numpy as np
import autograd as ag


In [3]:
# For simulations

# u = UberDatasetCreatorHet(drivers=1000, time_periods=30, regimes=3)

# df, mw, [beta0, beta1], y_sd = u.construct(
#     seed=1,
#     output_true_beta=True,
#     output_sigma=True,
#     y_sd=[1, 1, 1],
#     beta1=[-0.6, 0.2, 0.7],
#     beta0=[1, 2, 3],
#     weight=0.5,
# )

# beta_regime_0 = np.array([beta0[0], beta1[0]])[:, np.newaxis]
# beta_regime_1 = np.array([beta0[1], beta1[1]])[:, np.newaxis]
# beta_regime_2 = np.array([beta0[2], beta1[2]])[:, np.newaxis]

In [4]:
# For simulations
# R = df["regime"].values[:, np.newaxis]
# y = df["y"].values[:, np.newaxis]

# W0 = np.concatenate(
#     [np.ones((u.drivers * u.time_periods, 1)), df["drought_0"].values.reshape(-1, 1)],
#     axis=1,
# )  # add intercept
# W1 = np.concatenate(
#     [np.ones((u.drivers * u.time_periods, 1)), df["drought_1"].values.reshape(-1, 1)],
#     axis=1,
# )
# W2 = np.concatenate(
#     [np.ones((u.drivers * u.time_periods, 1)), df["drought_2"].values.reshape(-1, 1)],
#     axis=1,
# )

## IRLS

In [5]:
drop_date = "'2017-01'< date <='2019-12' and not '2018-01'<=date<='2018-07'"

lag_columns = [
    "intensity",
    "neg_intensity",
    "ind_low_intensity",
    "ind_intensity",
    "ind_pos_low_intensity",
    "ind_pos_intensity",
]

reg_data = RegressionData(
    outlier_quantile=0.99,
    drought_low=-1,
    drought_high=-1.5,
    lags=[
        1,
    ],
    lag_columns=lag_columns,
    drop_date=drop_date,
    driver_path="/Users/lordflaron/Documents/uganda-uber-paper/paper/data/uber_data/uganda_data/processed/drivers_all_hours.csv",
    weather_path="/Users/lordflaron/Documents/uganda-uber-paper/paper/data/weather/data/asap_data/raw/country_209_var_20_set_1_class_1_sensor_3.csv",
)

In [6]:
class MLSwitchingRegIRLS:
    
    def __init__(self, y, W, pi, R):
        self.y = y
        self.W = W
        self.pi = pi
        self.R = R
        
        self.num_params = self.W[0].shape[1]  # change if different number of parameters per regime
        self.n_regimes = len(np.unique(self.R))
        
    # def yWb(self, beta):
        
    def Pr(self, r, r_hat, sigma2, beta):
        
        W_mat = np.array(self.W)
        beta = beta.reshape(self.n_regimes, self.num_params, 1)
        
        regimes = np.exp(-((self.y.T - np.einsum("ijk,ikl -> ij", W_mat, beta)) ** 2)/2*sigma2)

        F = np.array([self.pi[:, i][np.newaxis, :] @ regimes for i in range(len(self.W))]).squeeze()

        cond_prob = self.pi[r_hat, r] * regimes[r] / F[r_hat]

        return cond_prob[:, np.newaxis]

    def _RR(self, r, r_hat, sigma2, beta):
        return np.where(self.R == r_hat, 1, 0) * self.Pr(
            r, r_hat, sigma2, beta
        )
    
    def RR_hat(self, beta, sigma2):
        RR_hat = np.zeros((len(self.W), len(self.W[0])))
        for r in range(len(self.W)):
            for r_hat in range(len(self.W)):
                RR_hat[r, :] += self._RR(
                    r, r_hat, sigma2, beta
                ).squeeze()
        return RR_hat
    
    def beta_h(self, r, rrh):
        return np.linalg.inv(
            np.linalg.multi_dot([self.W[r].T, np.diag(rrh[r].flatten()), self.W[r]])
        ).dot(np.linalg.multi_dot([self.W[r].T, np.diag(rrh[r].flatten()), self.y]))
        
    def sigma2_h(self, rrh, beta):
        W_mat = np.array(self.W)
        beta = beta.reshape(self.n_regimes, self.num_params, 1)
            
        sigma2_hat = np.sum(rrh * ((self.y.T - np.einsum("ijk,ikl -> ij", W_mat, beta)) ** 2))
        
        sigma2_hat = sigma2_hat / W_mat.shape[1]

        return sigma2_hat
    
    def jacobian(self, theta_old):
        rrh = self.RR_hat(theta_old[:-1], theta_old[-1])
                
        beta_hat = [self.beta_h(i, rrh) for i in range(self.n_regimes)]
        
        sigma2_hat = self.sigma2_h(rrh, theta_old[:-1])

        theta = np.vstack([
            np.array(beta_hat).flatten().reshape(self.num_params * self.n_regimes, 1),
            sigma2_hat,
        ])
        
        return theta - theta_old, theta
        
    def _irls(self, beta_0, sigma2_0,  
              max_iter=1000, tol=1e-6):
        
        theta_old = np.vstack([beta_0, sigma2_0])

        for i in range(max_iter):
            print(f"Starting iteration {i} with theta: {theta_old}")

            # calculate jacobian and get new theta
            jac, theta = self.jacobian(theta_old)
            
            norm = np.linalg.norm(jac)
                        
            print(f"Norm: {norm}")
            if norm < tol:
                theta_print = [f"{t:.3f}" for t in theta.flatten()]
                print(
                    f"Found in {i} iterations; with theta: {theta_print}, and norm: {norm}"
                )
                break
            theta_old = theta.copy()
        return theta
    
    def fit(self, beta_0, sigma2_0, max_iter=1000, tol=1e-6):
        return self._irls(beta_0, sigma2_0, max_iter, tol)

In [7]:
gaul_region_map_long_name = {
    "south highlands": "West",
    "west highlands": "West",
    "lake albert": "West",
    "eastern": "East",
    "west nile": "North",
    "mid northen": "North",
    "karamoja": "North",
    "lake victoria": "Central",
    "south drylands": "West",
    "south eastern": "East",
}


df_WMON = reg_data.regression_df(freq="W-MON")
df = reg_data.regression_df(freq="M")
df_2M = reg_data.regression_df(freq="2M")
df_3M = reg_data.regression_df(freq="BQ-JAN")

df_all_gaul = (
    reg_data.regression_df(freq="M", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_all_gaul_2M = (
    reg_data.regression_df(freq="2M", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_all_gaul_3M = (
    reg_data.regression_df(freq="BQ-JAN", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_all_gaul_WMON = (
    reg_data.regression_df(freq="W-MON", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_gaul = df.reset_index().merge(df_all_gaul, on="date")

df_gaul_2M = df_2M.reset_index().merge(df_all_gaul_2M, on="date")

df_gaul_WMON = df_WMON.reset_index().merge(df_all_gaul_WMON, on="date")

df_gaul_3M = df_3M.reset_index().merge(df_all_gaul_3M, on="date")

/Users/lordflaron/Documents/uganda-uber-paper/paper/scripts/regression.py:520: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda x: x.fillna(method='pad'))
/Users/lordflaron/Documents/uganda-uber-paper/paper/scripts/regression.py:520: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .appl

In [9]:
formula = lambda w: formulaic.Formula(
    f"np.arcsinh(hours_online) ~ -1 + `{w}` + shared_vehicle_1_0"
)


def irls_data(formula, df):
    fe = pyhdfe.create(
        df[["hashed_driver_uuid", "month", "year"]].values,
        drop_singletons=True,
    )

    singletons = fe.singleton_indices

    gauls = [
        "eastern",
        "karamoja",
        "lake albert",
        "lake victoria",
        "mid northen",
        "south eastern",
        "south drylands",
        "south highlands",
        "west nile",
        "west highlands",
    ]

    y = formula("eastern").get_model_matrix(df).lhs
    y_demeaned = fe.residualize(y)

    W = []
    W_demeaned = []

    for r in gauls:
        W.append(formula(r).get_model_matrix(df).rhs)
        W_demeaned.append(fe.residualize(formula(r).get_model_matrix(df).rhs))

    R = pd.Categorical(
        df.gaul_class,
        categories=gauls,
    ).codes[:, np.newaxis][~singletons]  # also drop the singletons

    x0 = np.ones((W[0].shape[1] * len(W), 1))
    
    mod = MLSwitchingRegIRLS(y = y_demeaned, 
                             W = W_demeaned, 
                             pi=cm_10,
                             R = R)

    return mod.fit(x0, 1)


In [24]:
# keep for monthly results
params_M = irls_data(formula, df_gaul)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
Norm: 7.884475658007424
Starting iteration 1 with theta: [[-0.12620314]
 [-0.65587648]
 [-0.22916397]
 [-0.68570789]
 [-0.31497129]
 [-0.55571025]
 [ 0.29378105]
 [-0.42337632]
 [-0.25105313]
 [-0.25383047]
 [-0.21873523]
 [-0.33060888]
 [ 0.39751931]
 [-0.44320473]
 [-0.23465957]
 [-0.48719055]
 [-0.21356889]
 [-0.59746489]
 [ 0.15956771]
 [-0.52278583]
 [ 6.24435523]]
Norm: 0.9167764599345013
Starting iteration 2 with theta: [[-0.12911253]
 [-0.67483967]
 [ 0.18998473]
 [-0.4423999 ]
 [-0.02498111]
 [-0.43444844]
 [-0.02885523]
 [-0.4620578 ]
 [ 0.01552474]
 [-0.14857953]
 [-0.1466965 ]
 [-0.33110377]
 [ 0.12614801]
 [-0.4006257 ]
 [ 0.18130529]
 [-0.27983688]
 [-0.17240031]
 [-0.61287671]
 [ 0.176792  ]
 [-0.5045223 ]
 [ 6.38387519]]
Norm: 0.1769171109518663
Starting iteration 3 with theta: [[-0.14295948]
 [-0.68508061]
 [ 0.

array([[-0.14262299],
       [-0.68426358],
       [ 0.10552228],
       [-0.48243913],
       [-0.08429944],
       [-0.46465378],
       [-0.09667865],
       [-0.48172533],
       [-0.00903943],
       [-0.16338423],
       [-0.16576559],
       [-0.33650808],
       [ 0.08195458],
       [-0.4252586 ],
       [ 0.13427573],
       [-0.31727318],
       [-0.18035455],
       [-0.61697671],
       [ 0.14750815],
       [-0.52271388],
       [ 6.38397723]])

In [28]:
## 2 month results
params_2M = irls_data(formula, df_gaul_2M)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
Norm: 6.541560874952173
Starting iteration 1 with theta: [[-0.22706023]
 [-0.64081635]
 [-1.4124897 ]
 [-0.57346224]
 [-0.33221738]
 [-0.43837658]
 [ 0.37694118]
 [-0.39231612]
 [-0.56969075]
 [-0.18590372]
 [-0.51774892]
 [-0.37412261]
 [ 0.29295645]
 [-0.43683759]
 [-0.20440289]
 [-0.44731338]
 [-0.19664796]
 [-0.58185827]
 [ 0.24897388]
 [-0.39594767]
 [ 2.87322132]]
Norm: 1.9732272723252249
Starting iteration 2 with theta: [[-0.18949557]
 [-0.75908007]
 [ 0.344769  ]
 [-0.46074681]
 [ 0.04470149]
 [-0.38031239]
 [-0.04712677]
 [-0.41234116]
 [-0.24976191]
 [-0.21829287]
 [-0.35884421]
 [-0.36113345]
 [ 0.07687867]
 [-0.39526665]
 [ 0.27516084]
 [-0.28386261]
 [-0.2734218 ]
 [-0.65846775]
 [ 0.19769843]
 [-0.41340045]
 [ 2.93028629]]
Norm: 0.9845302119169481
Starting iteration 3 with theta: [[-0.23448082]
 [-0.76800492]
 [-0.

In [13]:
# weekly results

params_WMON = irls_data(formula, df_gaul_WMON)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]


: 

In [10]:
# 3 month results

params_3M = irls_data(formula, df_gaul_3M)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
Norm: 6.166533602884685
Starting iteration 1 with theta: [[ 0.37363742]
 [-0.4847784 ]
 [-0.55721889]
 [-0.57571685]
 [ 0.39436532]
 [-0.31849151]
 [ 0.30010547]
 [-0.27692502]
 [ 0.13994819]
 [-0.36720378]
 [ 0.34770812]
 [-0.25483065]
 [ 0.50215364]
 [-0.43807601]
 [ 1.50510719]
 [-0.92724459]
 [-1.43767016]
 [-0.57321092]
 [ 0.50895344]
 [-0.33831146]
 [ 3.22532045]]
Norm: 8.950022158313663
Starting iteration 2 with theta: [[ 0.71924939]
 [-0.89395855]
 [-4.72858534]
 [-1.2842281 ]
 [ 0.7092453 ]
 [ 0.02163477]
 [ 0.11519088]
 [ 0.08142621]
 [-0.27507791]
 [-0.05628258]
 [ 0.4759516 ]
 [ 0.26457908]
 [ 1.11454502]
 [-0.58811928]
 [ 3.03295624]
 [-1.96819858]
 [-8.83913571]
 [-1.01558166]
 [ 0.45332443]
 [ 0.02280225]
 [ 4.65878217]]
Norm: 4.15593009279157
Starting iteration 3 with theta: [[ 1.85588784e+00]
 [-1.42288434e+00]


KeyboardInterrupt: 